# Sphere Model - FENDL-3.2c

This notebook compares YAMC and OpenMC simulation results for a simple sphere geometry using the FENDL-3.2c nuclear data library.

## Model Description

- **Geometry**: 35 cm radius sphere with vacuum boundary
- **Source**: Point source at origin, 14.06 MeV monoenergetic neutrons, isotropic
- **Material**: Single nuclide at 1.0 g/cm³, temperature 294 K
- **Statistics**: 500,000 particles × 10 batches = 5,000,000 total histories
- **Tallies**: Flux, heating, and all available reaction MTs (energy-integrated and energy-binned)

In [ ]:
# Setup and imports
import numpy as np
import pandas as pd
import glob
import os
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

# Configuration
OPENMC_DATA_DIR = {
    "fendl-3.2c": "/home/jon/nuclear_data/fendl-3.2c-hdf5/",
}
YAMC_NUCLEAR_DATA_KEYWORD = "fendl-3.2c"

NUCLEAR_DATA = "fendl-3.2c"
PARTICLES = 500000
BATCHES = 10
SOURCE_ENERGY = 14.06e6  # eV
MAX_PLOT_ENERGY = SOURCE_ENERGY * 1.1  # Source energy + 10%
FLUX_DECADES = 6  # Number of decades to show on flux y-axis (from max value)

In [ ]:
def run_openmc_simulation(nuclide, nuclear_data):
    """Run OpenMC simulation for a single nuclide."""
    import openmc as mc
    
    mc.config['cross_sections'] = f"{OPENMC_DATA_DIR[nuclear_data]}cross_sections.xml"
    
    # Geometry
    sphere1 = mc.Sphere(
        surface_id=2,
        x0=0.0, y0=0.0, z0=0.0,
        r=35.0,
        boundary_type='vacuum',
    )
    region1 = -sphere1
    
    # Material
    material1 = mc.Material()
    material1.material_id = 1
    material1.add_nuclide(nuclide, 1.0)
    material1.set_density("g/cm3", 1.0)
    material1.temperature = 294
    
    # Cell
    cell1 = mc.Cell(
        cell_id=1,
        name="inner_sphere",
        region=region1,
        fill=material1
    )
    geometry = mc.Geometry([cell1])
    
    # Get available MTs
    in_neu = mc.data.IncidentNeutron.from_hdf5(f"{OPENMC_DATA_DIR[nuclear_data]}neutron/{nuclide}.h5")
    mts_int = [mt for mt in in_neu.reactions.keys() if mt not in (1, 3, 4, 27, 101)]
    mts = [str(mt) for mt in mts_int]
    
    # Filters and tallies
    cell_filter = mc.CellFilter(cell1)
    energy_filter = mc.EnergyFilter.from_group_structure('VITAMIN-J-175')
    
    # Get energy bin boundaries - OpenMC returns (n_bins, 2) array with [low, high]
    energy_bins_2d = energy_filter.bins
    energy_bins = np.concatenate([[energy_bins_2d[0, 0]], energy_bins_2d[:, 1]])
    
    tallies = []
    
    all_scores = mts + ['total', 'absorption', 'flux', 'heating', 'heating-local', 'H1-production', 'H2-production', 'H3-production', 'He3-production', 'He4-production']
    for score in all_scores:                                                                                                                                
        tally = mc.Tally()                                                                                                                                  
        tally.filters = [cell_filter]                                                                                                                       
        tally.scores = [score]                                                                                                                              
        tally.name = str(score)                                                                                                                             
        tallies.append(tally)    
    
    # Energy-binned flux tally
    tally_spectrum = mc.Tally()
    tally_spectrum.filters = [cell_filter, energy_filter]
    tally_spectrum.scores = ['flux']
    tally_spectrum.name = "flux_energy_binned"
    tallies.append(tally_spectrum)
    
    # Source and settings
    source = mc.IndependentSource(
        angle=mc.stats.Isotropic(),
        energy=mc.stats.Discrete([SOURCE_ENERGY], [1.0]),
        space=mc.stats.Point([0, 0, 0])
    )
    settings = mc.Settings(
        particles=PARTICLES,
        batches=BATCHES,
        source=source,
        seed=1,
        run_mode='fixed source',
        output={'tallies': False, 'summary': False}
    )
    
    # Run
    model = mc.Model(geometry=geometry, settings=settings, tallies=tallies)
    sp_filename = model.run(apply_tally_results=True)
    sp = mc.StatePoint(sp_filename)
    
    # Collect results
    transport_time = sp.runtime['transport']
    total_particles = PARTICLES * BATCHES
    particles_per_second = total_particles / transport_time
    
    results = {
        'nuclide': nuclide,
        'code': 'openmc',
        'particles/s': particles_per_second,
        'tallies': {},
        'tally_std': {}
    }
    
    for tally in tallies:
        if tally.name != "flux_energy_binned":
            results['tallies'][tally.name] = tally.mean.flatten()[0]
            results['tally_std'][tally.name] = tally.std_dev.flatten()[0]
    
    # Get spectrum tally
    results['flux_spectra_mean'] = np.array(tally_spectrum.mean).squeeze()
    results['flux_spectra_std_dev'] = np.array(tally_spectrum.std_dev).squeeze()
    results['energy_bins'] = energy_bins
    
    return results

In [ ]:
def run_yamc_simulation(nuclide, nuclear_data):
    """Run YAMC simulation for a single nuclide. Returns (results_dict, model)."""
    import yamc as mc
    
    # Geometry
    sphere1 = mc.Sphere(
        surface_id=2,
        x0=0.0, y0=0.0, z0=0.0,
        r=35.0,
        boundary_type='vacuum',
    )
    region1 = -sphere1
    
    # Material
    material1 = mc.Material()
    material1.material_id = 1
    material1.add_nuclide(nuclide, 1.0)
    material1.set_density("g/cm3", 1.0)
    material1.temperature = 294
    material1.read_nuclear_data(YAMC_NUCLEAR_DATA_KEYWORD)
    
    # Get available MTs
    mts = [mt for mt in material1.reaction_mts if mt not in (1, 3, 4, 27, 101)]
    
    # Cell
    cell1 = mc.Cell(
        cell_id=1,
        name="inner_sphere",
        region=region1,
        fill=material1
    )
    geometry = mc.Geometry([cell1])
    
    # Filters and tallies
    cell_filter = mc.CellFilter(cell1)
    energy_filter = mc.EnergyFilter.from_group_structure('VITAMIN-J-175')
    
    # YAMC returns energy bins as 1D array of boundaries
    energy_bins = np.array(energy_filter.bins)
    
    tallies = []
    
    all_scores = mts + ['total', 'absorption', 'flux', 'heating', 'heating-local', 'H1-production', 'H2-production', 'H3-production', 'He3-production', 'He4-production']
    for score in all_scores:                                                                                                                                
        tally = mc.Tally()                                                                                                                                  
        tally.filters = [cell_filter]                                                                                                                       
        tally.scores = [score]                                                                                                                              
        tally.name = str(score)                                                                                                                             
        tallies.append(tally)     
    
    # Energy-binned flux tally
    tally_spectrum = mc.Tally()
    tally_spectrum.filters = [cell_filter, energy_filter]
    tally_spectrum.scores = ['flux']
    tally_spectrum.name = "flux_energy_binned"
    tallies.append(tally_spectrum)
    
    # Source and settings
    source = mc.IndependentSource(
        angle=mc.stats.Isotropic(),
        energy=mc.stats.Discrete([SOURCE_ENERGY], [1.0]),
        space=mc.stats.Point([0, 0, 0])
    )
    settings = mc.Settings(
        particles=PARTICLES,
        batches=BATCHES,
        source=source,
        seed=1
    )
    
    # Run
    model = mc.Model(geometry=geometry, settings=settings, tallies=tallies)
    model.run()
    
    # Collect results
    results = {
        'nuclide': nuclide,
        'code': 'yamc',
        'particles/s': model.particles_per_second,
        'tallies': {},
        'tally_std': {}
    }
    
    for tally in tallies:
        if tally.name != "flux_energy_binned":
            results['tallies'][tally.name] = tally.mean[0]
            results['tally_std'][tally.name] = tally.std_dev[0]
    
    # Get spectrum tally
    results['flux_spectra_mean'] = np.array(tally_spectrum.mean).squeeze()
    results['flux_spectra_std_dev'] = np.array(tally_spectrum.std_dev).squeeze()
    results['energy_bins'] = energy_bins
    
    return results, model

In [ ]:
def create_step_trace(energy_bins, values, name, color, dash=None):
    """Create a step plot trace for Plotly (similar to matplotlib step with where='post')."""
    # For step plot: duplicate x values and create staircase y values
    x_step = []
    y_step = []
    for i in range(len(values)):
        x_step.extend([energy_bins[i], energy_bins[i+1]])
        y_step.extend([values[i], values[i]])
    
    return go.Scatter(
        x=x_step,
        y=y_step,
        mode='lines',
        name=name,
        line=dict(color=color, width=2, dash=dash),
    )


def create_comparison_plot(openmc_results, yamc_results):
    """Create a Plotly subplot with flux spectra (top) and ratio (bottom)."""
    
    energy_bins = openmc_results['energy_bins']
    # Geometric mean for bin centers (better for log scale)
    bin_centers = np.sqrt(energy_bins[:-1] * energy_bins[1:])
    
    openmc_flux = openmc_results['flux_spectra_mean'].copy()
    openmc_std = openmc_results['flux_spectra_std_dev'].copy()
    yamc_flux = yamc_results['flux_spectra_mean'].copy()
    yamc_std = yamc_results['flux_spectra_std_dev'].copy()

    # Normalize by lethargy bin width for per-unit-lethargy spectrum
    # Equivalent to openmc.EnergyFilter.lethargy_bin_width / yamc.EnergyFilter.lethargy_bin_width
    lethargy_bin_width = np.log10(energy_bins[1:] / energy_bins[:-1])
    openmc_flux /= lethargy_bin_width
    openmc_std /= lethargy_bin_width
    yamc_flux /= lethargy_bin_width
    yamc_std /= lethargy_bin_width

    # Find the lowest energy bin with non-zero flux in either code
    has_flux = (openmc_flux > 0) | (yamc_flux > 0)
    if np.any(has_flux):
        first_nonzero_idx = np.argmax(has_flux)  # First True index
        min_plot_energy = energy_bins[first_nonzero_idx]
    else:
        min_plot_energy = energy_bins[0]
    
    # Calculate flux threshold: values below max/10^FLUX_DECADES are set to NaN
    # This removes noise floor while letting y-axis auto-scale to actual data
    max_flux = max(
        np.max(openmc_flux[openmc_flux > 0]) if np.any(openmc_flux > 0) else 1e-10,
        np.max(yamc_flux[yamc_flux > 0]) if np.any(yamc_flux > 0) else 1e-10
    )
    flux_threshold = max_flux / (10 ** FLUX_DECADES)
    
    # Mask values below threshold for plotting (keeps original for ratio calc)
    openmc_flux_plot = np.where(openmc_flux >= flux_threshold, openmc_flux, np.nan)
    yamc_flux_plot = np.where(yamc_flux >= flux_threshold, yamc_flux, np.nan)
    openmc_std_plot = np.where(openmc_flux >= flux_threshold, openmc_std, np.nan)
    yamc_std_plot = np.where(yamc_flux >= flux_threshold, yamc_std, np.nan)
    
    # Calculate ratio with error propagation (use original data)
    valid = (openmc_flux > 0) & (yamc_flux > 0)
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.where(valid, yamc_flux / openmc_flux, np.nan)
        rel_err_yamc = np.where(yamc_flux > 0, yamc_std / yamc_flux, 0)
        rel_err_openmc = np.where(openmc_flux > 0, openmc_std / openmc_flux, 0)
    ratio_rel_err = np.sqrt(rel_err_yamc**2 + rel_err_openmc**2)
    ratio_std = np.where(valid, ratio * ratio_rel_err, np.nan)
    
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        row_heights=[0.7, 0.3],
        subplot_titles=('Flux per Unit Lethargy', 'YAMC / OpenMC Ratio')
    )
    
    # Top plot: Flux spectra as step plots (using masked data)
    fig.add_trace(
        create_step_trace(energy_bins, openmc_flux_plot, 'OpenMC', 'blue'),
        row=1, col=1
    )
    
    fig.add_trace(
        create_step_trace(energy_bins, yamc_flux_plot, 'YAMC', 'red', dash='dash'),
        row=1, col=1
    )
    
    # Add error bars at bin centers (using masked data)
    fig.add_trace(
        go.Scatter(
            x=bin_centers,
            y=openmc_flux_plot,
            mode='markers',
            name='OpenMC error',
            marker=dict(color='blue', size=4, symbol='circle'),
            error_y=dict(type='data', array=openmc_std_plot, visible=True, color='blue', thickness=1),
            showlegend=False,
        ),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Scatter(
            x=bin_centers,
            y=yamc_flux_plot,
            mode='markers',
            name='YAMC error',
            marker=dict(color='red', size=4, symbol='square'),
            error_y=dict(type='data', array=yamc_std_plot, visible=True, color='red', thickness=1),
            showlegend=False,
        ),
        row=1, col=1
    )
    
    # Bottom plot: Ratio as step plot with uncertainty band
    fig.add_trace(
        create_step_trace(energy_bins, ratio, 'YAMC/OpenMC', 'green'),
        row=2, col=1
    )
    
    # Add uncertainty band using fill
    fig.add_trace(
        go.Scatter(
            x=np.concatenate([bin_centers, bin_centers[::-1]]),
            y=np.concatenate([ratio + ratio_std, (ratio - ratio_std)[::-1]]),
            fill='toself',
            fillcolor='rgba(0, 128, 0, 0.2)',
            line=dict(color='rgba(255,255,255,0)'),
            name='Uncertainty',
            showlegend=False,
        ),
        row=2, col=1
    )
    
    # Reference line at 1.0
    fig.add_hline(y=1.0, line_dash="dash", line_color="gray", row=2, col=1)
    
    # Update axes with scientific notation format and x-axis limits
    fig.update_xaxes(
        type="log",
        title_text="Energy (eV)",
        range=[np.log10(min_plot_energy), np.log10(MAX_PLOT_ENERGY)],
        exponentformat='e',
        row=2, col=1
    )
    fig.update_xaxes(
        type="log",
        range=[np.log10(min_plot_energy), np.log10(MAX_PLOT_ENERGY)],
        exponentformat='e',
        row=1, col=1
    )
    fig.update_yaxes(
        type="log",
        title_text="Flux / Lethargy",
        exponentformat='e',
        row=1, col=1
    )
    fig.update_yaxes(
        title_text="Ratio",
        range=[0.5, 1.5],
        row=2, col=1
    )
    
    fig.update_layout(
        height=700,
        legend=dict(x=0.02, y=0.98),
        margin=dict(l=60, r=20, t=40, b=40)
    )
    
    return fig


def calculate_spectral_agreement(openmc_results, yamc_results, flux_threshold=1e-6):
    """Calculate statistical agreement metrics for flux spectra comparison."""
    
    openmc_flux = openmc_results['flux_spectra_mean']
    openmc_std = openmc_results['flux_spectra_std_dev']
    yamc_flux = yamc_results['flux_spectra_mean']
    yamc_std = yamc_results['flux_spectra_std_dev']
    
    chi_squared_terms = []
    rel_diffs = []
    weighted_rel_diffs = []
    flux_weights = []
    
    for i in range(len(openmc_flux)):
        omc = openmc_flux[i]
        ymc = yamc_flux[i]
        omc_std = openmc_std[i]
        ymc_std = yamc_std[i]
        
        # Chi-squared term: (observed - expected)^2 / (sigma_obs^2 + sigma_exp^2)
        combined_std = np.sqrt(omc_std**2 + ymc_std**2)
        if combined_std > 0:
            chi_squared_terms.append(((ymc - omc) / combined_std)**2)
        
        # Relative difference (only for bins with significant flux)
        if omc > flux_threshold or ymc > flux_threshold:
            if omc > 0:
                rel_diff = abs((ymc - omc) / omc) * 100
                if np.isfinite(rel_diff):
                    rel_diffs.append(rel_diff)
                    weight = max(omc, ymc)
                    weighted_rel_diffs.append(rel_diff * weight)
                    flux_weights.append(weight)
    
    # Calculate metrics
    chi_squared = np.sum(chi_squared_terms)
    dof = len(chi_squared_terms)
    reduced_chi_squared = chi_squared / dof if dof > 0 else 0.0
    
    median_rel_diff = np.median(rel_diffs) if rel_diffs else 0.0
    weighted_avg_rel_diff = np.sum(weighted_rel_diffs) / np.sum(flux_weights) if flux_weights else 0.0
    
    # Quality assessment
    if reduced_chi_squared < 1.5:
        quality = "Excellent"
    elif reduced_chi_squared < 3.0:
        quality = "Good"
    elif reduced_chi_squared < 5.0:
        quality = "Fair"
    else:
        quality = "Poor"
    
    return {
        'reduced_chi_squared': reduced_chi_squared,
        'degrees_of_freedom': dof,
        'median_rel_diff': median_rel_diff,
        'weighted_avg_rel_diff': weighted_avg_rel_diff,
        'quality': quality,
        'bins_analyzed': len(rel_diffs),
        'total_bins': len(openmc_flux)
    }


def create_agreement_table(stats):
    """Create a summary table of spectral agreement statistics."""
    rows = [
        {'Metric': 'Reduced Chi-Squared', 'Value': f"{stats['reduced_chi_squared']:.2f}", 
         'Note': f"(ideal ~ 1.0, dof={stats['degrees_of_freedom']})"},
        {'Metric': 'Weighted Avg Rel. Diff', 'Value': f"{stats['weighted_avg_rel_diff']:.2f}%",
         'Note': '(flux-weighted)'},
        {'Metric': 'Median Rel. Diff', 'Value': f"{stats['median_rel_diff']:.2f}%",
         'Note': '(robust to outliers)'},
        {'Metric': 'Agreement Quality', 'Value': stats['quality'],
         'Note': f"({stats['bins_analyzed']}/{stats['total_bins']} bins analyzed)"},
    ]
    return pd.DataFrame(rows)


# MT number to reaction description mapping from YAMC
import yamc
MT_DESCRIPTIONS = yamc.reaction_names()

# Score name to MT number mapping (reverse lookup + special score aliases)
SCORE_TO_MT = {name: mt for mt, name in MT_DESCRIPTIONS.items()}
SCORE_TO_MT.update({
    'total': 1,
    'elastic': 2,
    'absorption': 27,
    'fission': 18,
    'heating': 301,
    'heating-local': 901,
    'H1-production': 203,
    'H2-production': 204,
    'H3-production': 205,
    'He3-production': 206,
    'He4-production': 207,
})


def get_mt_and_reaction(score_name):
    """Get MT number and reaction description for a score.
    
    Returns (mt_display, reaction_description) tuple.
    """
    # Try to parse as integer MT number
    try:
        mt = int(score_name)
        return str(mt), MT_DESCRIPTIONS.get(mt, f'MT {mt}')
    except ValueError:
        pass
    
    # Check if it's a known score name with an MT equivalent
    if score_name in SCORE_TO_MT:
        mt = SCORE_TO_MT[score_name]
        return str(mt), score_name
    
    # Unknown non-numeric score (flux, production scores, etc.)
    return '-', score_name


def create_tally_table(openmc_results, yamc_results):
    """Create a comparison table of all tallies, deduplicated by MT number and sorted by MT."""

    # Get all tally names from both codes
    all_tallies = set(openmc_results['tallies'].keys()) | set(yamc_results['tallies'].keys())

    # Group tallies by MT number to deduplicate (e.g., "1" and "total" both map to MT 1)
    mt_to_scores = {}  # mt_display -> list of score names
    for name in all_tallies:
        mt_display, _ = get_mt_and_reaction(name)
        if mt_display not in mt_to_scores:
            mt_to_scores[mt_display] = []
        mt_to_scores[mt_display].append(name)

    rows = []

    for mt_display, score_names in mt_to_scores.items():
        # For each MT, find the best data from the available score names
        # Prefer data that has valid (non-zero, non-nan) values
        best_openmc_val = np.nan
        best_openmc_std = np.nan
        best_yamc_val = np.nan
        best_yamc_std = np.nan
        best_reaction = None

        for name in score_names:
            openmc_val = openmc_results['tallies'].get(name, np.nan)
            openmc_std = openmc_results['tally_std'].get(name, np.nan)
            yamc_val = yamc_results['tallies'].get(name, np.nan)
            yamc_std = yamc_results['tally_std'].get(name, np.nan)

            _, reaction = get_mt_and_reaction(name)

            # Update OpenMC value if this one is better (non-nan and non-zero preferred)
            if not np.isnan(openmc_val) and openmc_val != 0:
                if np.isnan(best_openmc_val) or best_openmc_val == 0:
                    best_openmc_val = openmc_val
                    best_openmc_std = openmc_std
            elif np.isnan(best_openmc_val):
                best_openmc_val = openmc_val
                best_openmc_std = openmc_std

            # Update YAMC value if this one is better
            if not np.isnan(yamc_val) and yamc_val != 0:
                if np.isnan(best_yamc_val) or best_yamc_val == 0:
                    best_yamc_val = yamc_val
                    best_yamc_std = yamc_std
            elif np.isnan(best_yamc_val):
                best_yamc_val = yamc_val
                best_yamc_std = yamc_std

            # Use the reaction description from MT_DESCRIPTIONS if available
            if best_reaction is None:
                best_reaction = reaction

        # Get the canonical reaction name from MT_DESCRIPTIONS
        if mt_display != '-':
            try:
                mt_num = int(mt_display)
                best_reaction = MT_DESCRIPTIONS.get(mt_num, best_reaction)
            except ValueError:
                pass

        # Skip rows where both codes report zero (or nan)
        openmc_is_zero = best_openmc_val == 0 or np.isnan(best_openmc_val)
        yamc_is_zero = best_yamc_val == 0 or np.isnan(best_yamc_val)
        if openmc_is_zero and yamc_is_zero:
            continue

        if best_openmc_val != 0 and not np.isnan(best_openmc_val):
            rel_diff = (best_yamc_val - best_openmc_val) / best_openmc_val * 100
        else:
            rel_diff = np.nan

        rows.append({
            'MT': mt_display,
            'Reaction': best_reaction,
            'OpenMC': f"{best_openmc_val:.4e} +/- {best_openmc_std:.2e}",
            'YAMC': f"{best_yamc_val:.4e} +/- {best_yamc_std:.2e}",
            'Rel. Diff (%)': rel_diff,
            '_mt_sort': int(mt_display) if mt_display != '-' else 9999
        })

    # Sort by MT number (numeric sort, with non-MT scores at the end)
    rows.sort(key=lambda x: x['_mt_sort'])

    # Remove internal column and format
    for row in rows:
        del row['_mt_sort']
        if not np.isnan(row['Rel. Diff (%)']):
            row['Rel. Diff (%)'] = f"{row['Rel. Diff (%)']:.2f}%"
        else:
            row['Rel. Diff (%)'] = 'N/A'

    return pd.DataFrame(rows)

In [ ]:
# Get list of nuclides
h5_files = sorted(glob.glob(os.path.join(OPENMC_DATA_DIR[NUCLEAR_DATA], 'neutron', "*.h5")))
nuclides = [os.path.splitext(os.path.basename(f))[0] for f in h5_files]

if not nuclides:
    raise FileNotFoundError(
        f"No HDF5 files found in {OPENMC_DATA_DIR[NUCLEAR_DATA]}neutron/\n"
        f"Please check that OPENMC_DATA_DIR['{NUCLEAR_DATA}'] points to a valid nuclear data directory."
    )

# For testing, limit to first few nuclides
# Comment out the next line to run all nuclides
nuclides = nuclides[:2]

print(f"Running simulations for {len(nuclides)} nuclides...")

In [ ]:
# Run all simulations and collect results
all_results = {}
first_yamc_model = None

for nuclide in nuclides:
    print(f"\nProcessing {nuclide}...")
    try:
        yamc_results, yamc_model = run_yamc_simulation(nuclide, NUCLEAR_DATA)
        openmc_results = run_openmc_simulation(nuclide, NUCLEAR_DATA)

        if first_yamc_model is None:
            first_yamc_model = yamc_model

        all_results[nuclide] = {
            'openmc': openmc_results,
            'yamc': yamc_results
        }
        print(f"  OpenMC: {openmc_results['particles/s']:.0f} particles/s")
        print(f"  YAMC:   {yamc_results['particles/s']:.0f} particles/s")
    except Exception as e:
        print(f"  Error: {e}")
        continue

print(f"\nCompleted {len(all_results)} / {len(nuclides)} nuclides successfully.")

In [ ]:
# Display sphere model geometry with source from first simulation
import base64
from IPython.display import HTML

if first_yamc_model is not None:
    html_plot = first_yamc_model.plot(basis="xz", n_samples=500, axis_units="cm")
    with open("sphere_model_plot.html", "w") as f:
        f.write(html_plot)
    encoded = base64.b64encode(html_plot.encode()).decode()
    display(HTML(f'<iframe src="data:text/html;base64,{encoded}" width="100%" height="600" frameborder="0"></iframe>'))
else:
    print("No successful YAMC simulation to plot.")

## Results by Nuclide

Below are the comparison results for each nuclide. Each section includes:

- Interactive flux spectra plot (log-log scale)
- YAMC/OpenMC ratio plot with uncertainty band
- Tally comparison table sorted by relative difference
- Performance comparison (particles/second)

In [ ]:
# Display results for each nuclide
for nuclide, results in all_results.items():
    openmc_results = results['openmc']
    yamc_results = results['yamc']
    
    # Header
    display(Markdown(f"## {nuclide}"))
    
    # Performance comparison
    openmc_pps = openmc_results['particles/s']
    yamc_pps = yamc_results['particles/s']
    speedup = yamc_pps / openmc_pps
    display(Markdown(f"**Performance:** OpenMC: {openmc_pps:,.0f} particles/s | YAMC: {yamc_pps:,.0f} particles/s | Speedup: {speedup:.2f}x"))
    
    # Flux spectra plot
    fig = create_comparison_plot(openmc_results, yamc_results)
    fig.update_layout(title=f"{nuclide} - Flux Spectra Comparison")
    fig.show()
    
    # Spectral agreement statistics
    display(Markdown("### Spectral Agreement"))
    stats = calculate_spectral_agreement(openmc_results, yamc_results)
    agreement_table = create_agreement_table(stats)
    display(agreement_table.style.hide(axis='index'))
    
    # Tally table
    display(Markdown("### Tally Comparison"))
    table = create_tally_table(openmc_results, yamc_results)
    display(table.style.hide(axis='index'))
    
    print()  # Add spacing between nuclides